# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [ ]:
import math
import re
import time
from numbers import Real
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

COLUMNS = ['UPC', 'Title', 'Price (£)', 'Rating', 'Genre', 'Availability', 'Description']
RATINGS = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}


def scrape_books(min_rating, max_price):
    """Return all books with rating >= min_rating and price <= max_price.

    Follow every catalogue page. Fetch details only for matching books.
    HTTP errors are raised so an incomplete extraction is never reported as complete.
    """
    for name, value in [('min_rating', min_rating), ('max_price', max_price)]:
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value):
            raise ValueError(f'{name} must be a finite number')
    if not 1 <= min_rating <= 5:
        raise ValueError('min_rating must be between 1 and 5')
    if max_price < 0:
        raise ValueError('max_price must be non-negative')

    def money(text):
        return float(re.sub(r'[^0-9.]', '', text))

    def get_soup(session, url):
        time.sleep(0.15)  # Make sequential, paced requests to this practice site.
        response = session.get(url, timeout=30)
        response.raise_for_status()
        response.encoding = 'utf-8'
        return BeautifulSoup(response.text, 'html.parser')

    rows = []
    seen_pages = set()
    seen_books = set()
    page_url = 'https://books.toscrape.com/'
    with requests.Session() as session:
        session.headers.update({'User-Agent': 'Ironhack-Web-Scraping-Educational-Lab/1.0'})
        while page_url:
            if page_url in seen_pages:
                raise RuntimeError('Repeated pagination URL')
            seen_pages.add(page_url)
            soup = get_soup(session, page_url)
            products = soup.select('article.product_pod')
            if not products:
                raise RuntimeError(f'No book cards found: {page_url}')
            for product in products:
                rating_tag = product.select_one('p.star-rating')
                rating = next((RATINGS[c] for c in rating_tag.get('class', []) if c in RATINGS), None)
                if rating is None:
                    raise ValueError('Unknown star-rating class')
                price = money(product.select_one('p.price_color').get_text(strip=True))
                if rating < min_rating or price > max_price:
                    continue
                book_url = urljoin(page_url, product.select_one('h3 a')['href'])
                if book_url in seen_books:
                    continue
                seen_books.add(book_url)
                detail = get_soup(session, book_url)
                info = {tr.th.get_text(strip=True): tr.td.get_text(' ', strip=True)
                        for tr in detail.select('table.table-striped tr') if tr.th and tr.td}
                breadcrumb = detail.select('ul.breadcrumb li a')
                description_heading = detail.select_one('#product_description')
                description = description_heading.find_next_sibling('p') if description_heading else None
                rows.append({
                    'UPC': info['UPC'],
                    'Title': detail.select_one('.product_main h1').get_text(strip=True),
                    'Price (£)': price,
                    'Rating': rating,
                    'Genre': breadcrumb[-1].get_text(strip=True) if breadcrumb else '',
                    'Availability': info.get('Availability', ''),
                    'Description': description.get_text(' ', strip=True) if description else ''
                })
            next_link = soup.select_one('li.next a')
            page_url = urljoin(page_url, next_link['href']) if next_link else None

    result = pd.DataFrame(rows, columns=COLUMNS)
    result = result.astype({'Price (£)': 'float64', 'Rating': 'int64'})
    return result.drop_duplicates(subset='UPC').reset_index(drop=True)


## Ejecución y comprobaciones
Instalar dependencias si es necesario: `%pip install requests beautifulsoup4 pandas`.
El filtro es inclusivo: 4 estrellas o más y precio de hasta £20. Se recorren todas las páginas. El género se conserva para poder filtrar el DataFrame después. Los datos del sitio son ficticios y sirven para practicar scraping.

In [ ]:
books = scrape_books(4.0, 20)
assert list(books.columns) == COLUMNS
assert books['Rating'].ge(4).all()
assert books['Price (£)'].le(20).all()
assert books['UPC'].is_unique
display(books)
print(f'Libros encontrados: {len(books)}')
books.to_csv('books_rating4_max20.csv', index=False, encoding='utf-8-sig')


In [ ]:
# Otros umbrales, sin repetir peticiones: son subconjuntos de la extracción anterior.
five_stars = books.loc[books['Rating'].ge(5)].reset_index(drop=True)
under_15 = books.loc[books['Price (£)'].le(15)].reset_index(drop=True)
display(five_stars, under_15)
# Caso sin coincidencias y validación de parámetros.
empty = scrape_books(5, 0)
assert empty.empty and list(empty.columns) == COLUMNS
for args in [(0, 20), (6, 20), (4, -1), (True, 20)]:
    try:
        scrape_books(*args)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Parámetros inválidos aceptados: {args}')
